# Conventional binned likelihood fit — leptonic channel

This notebook estimates the expected sensitivity to the longitudinal, transverse, and mixed signal components using the leptonic signal and background templates.

## Required input

Run `study_angles.ipynb` first. It creates `outputs/histos.root`, the template file read by this notebook. No original event-level ROOT samples are reopened here.

## Analysis performed

- One-, two-, and three-parameter fits for the signal-strength parameters `mu_00`, `mu_tt`, and `mu_mix`.
- Fit results and parameter correlations for the observables `cos(theta*)`, leading-lepton `pT`, and dilepton invariant mass `mll`.
- One- and two-dimensional likelihood scans, with the other signal-strength parameters fixed or profiled as appropriate.

The pseudo-data are Asimov data, constructed from the nominal signal and background templates. The fit is therefore an expected-sensitivity study, not a fit to observed collision data.

## How to run

After configuring the input samples and running `study_angles.ipynb`, open this notebook and execute all cells from top to bottom. Plots and scan figures are saved under `outputs/plots/` and `outputs/likelihood_plots/`.


In [ ]:
# Setup: ROOT, numerical libraries, and shared paths/settings.
import os
from pathlib import Path

import ROOT
import numpy as np
from iminuit import Minuit

from analysis_config import (
    COLLISION_ENERGY_GEV,
    LUMI_PB,
    TEMPLATE_FILE,
    output_dir,
    output_path,
    PROJECT_DIR,
)

# Keep relative TMVA-generated folders anchored to the project directory.
os.chdir(PROJECT_DIR)


In [ ]:
# Shared plot styling and ROOT graphics helpers.
# Text and line sizes are fractions of the pad height. Increase or decrease them
# here to adjust the readability of the exported figures.
SIZE_LABEL  = 0.045     # numeri sugli assi
SIZE_TITLE  = 0.050     # titoli degli assi
SIZE_LEGEND = 0.036     # testo delle legende
SIZE_FCC    = 0.045     # scritta "FCC-ee Simulation" (l'energia e' un po' piu' piccola)
LINE_WIDTH  = 3         # spessore delle linee degli istogrammi

# --- Impostazioni globali di ROOT ---
ROOT.TH1.AddDirectory(False)        # gli istogrammi non dipendono dai file aperti
ROOT.gStyle.SetOptStat(0)           # niente riquadro delle statistiche
ROOT.gStyle.SetOptTitle(0)          # niente titolo in cima ai grafici
ROOT.gStyle.SetPadTickX(1)          # tacche anche sull'asse superiore
ROOT.gStyle.SetPadTickY(1)          # tacche anche sull'asse destro
ROOT.gStyle.SetEndErrorSize(0)      # barre d'errore senza cappuccio
ROOT.gStyle.SetFrameLineWidth(2)    # assi piu' spessi
ROOT.gStyle.SetLabelFont(42, "xyz")
ROOT.gStyle.SetTitleFont(42, "xyz")
ROOT.gStyle.SetLabelSize(SIZE_LABEL, "xyz")
ROOT.gStyle.SetTitleSize(SIZE_TITLE, "xyz")
ROOT.gStyle.SetTitleOffset(1.4, "xy")

# --- Sample colours ---
COLORS = {
    "00":    ROOT.TColor.GetColor("#0072B2"),   # blu     -> longitudinale (LL)
    "TT":    ROOT.TColor.GetColor("#C0392B"),   # rosso   -> trasversa (TT)
    "mixed": ROOT.TColor.GetColor("#7B3294"),   # viola   -> mista (LT)
    "bkg":   ROOT.TColor.GetColor("#666666"),   # grigio  -> fondo
    "sig":   ROOT.TColor.GetColor("#000000"),   # nero    -> segnale totale
}

# --- Colori per fit e scan di likelihood ---
COLORS_FIT = {
    "profilato":        ROOT.TColor.GetColor("#2171B5"),   # altri parametri profilati
    "profilato_chiaro": ROOT.TColor.GetColor("#9ECAE1"),
    "fisso":            ROOT.TColor.GetColor("#000000"),   # altri parametri fissi a 1
    "fisso_chiaro":     ROOT.TColor.GetColor("#808080"),
    "soglie":           ROOT.TColor.GetColor("#808080"),   # linee a 1 e 2 sigma
}

# --- Etichette ---
Y_TITLE = "Events / bin"
LABELS = {
    "00": "Longitudinal (00)",
    "TT": "Transverse (TT)",
    "mixed": "Mixed (0T + T0)",
    "bkg": "Background",
    "sig": "Signal",
}

# --- Funzioni grafiche ---
def make_canvas(name, title="", width=900, height=650):
    """Create a canvas with the standard margins used for the analysis plots."""
    c = ROOT.TCanvas(name, title, width, height)
    c.SetLeftMargin(0.17)
    c.SetBottomMargin(0.18)
    c.SetTopMargin(0.12)
    c.SetRightMargin(0.05)
    return c


# Keep TLatex objects alive while the canvases are displayed.
_FCC_LABEL_OBJECTS = []

def draw_fcc_label():
    """Draw the FCC-ee Simulation label, collision energy, and integrated luminosity."""
    latex = ROOT.TLatex()
    latex.SetNDC()
    latex.SetTextFont(42)
    latex.SetTextAlign(11)                  # allineato a sinistra
    latex.SetTextSize(SIZE_FCC)
    # Use one string with two fonts: #font[61] is bold and #font[52] is italic.
    t1 = latex.DrawLatex(0.24, 0.92, "#font[61]{FCC-ee} #font[52]{#scale[0.85]{Simulation}}")
    latex.SetTextAlign(31)                  # allineato a destra, sul bordo del grafico
    latex.SetTextSize(SIZE_FCC * 0.78)
    t2 = latex.DrawLatex(0.95, 0.92, f"#sqrt{{s}} = {COLLISION_ENERGY_GEV:g} GeV, L = {LUMI_PB / 1e6:g} ab^{{-1}}")
    _FCC_LABEL_OBJECTS.extend([latex, t1, t2])


def style_hist(hist, color, line_style=1, fill=True):
    """Set the histogram line colour and width, with an optional transparent fill."""
    hist.SetLineColor(color)
    hist.SetLineWidth(LINE_WIDTH)
    hist.SetLineStyle(line_style)
    if fill:
        hist.SetFillColorAlpha(color, 0.15)


def make_legend(x1, y1, x2, y2, header=None):
    """Create a legend without a border or background, with an optional header."""
    leg = ROOT.TLegend(x1, y1, x2, y2)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    leg.SetTextFont(42)
    leg.SetTextSize(SIZE_LEGEND)
    leg.SetMargin(0.12)                     # meno spazio per il simbolo, piu' per il testo
    if header:
        leg.SetHeader(header)
    return leg


# One-, two-, and three-parameter likelihood fits

In [ ]:
# Fit 1D, 2D, and 3D across all observables
from itertools import combinations

PARAMS = ("mu_00", "mu_tt", "mu_mix")
VARIABILI = ("cos_theta_star", "lep_pt", "mll")


NOMI_PARAMETRI = {
    "mu_00": "μ_LL",
    "mu_tt": "μ_TT",
    "mu_mix": "μ_mix", 
}

NOMI_VARIABILI = {
    "cos_theta_star": r"cos(θ*)",
    "lep_pt": r"pT leptone",
    "mll": r"massa dileptonica mll",
}

try:
    import pandas as pd
    from IPython import get_ipython
    from IPython.display import display, Markdown
    IN_NOTEBOOK = get_ipython() is not None
except ImportError:
    pd = None
    IN_NOTEBOOK = False


def carica_template(var_name, root_file=TEMPLATE_FILE):
    """Load ROOT templates and return the background, LL, TT, and mixed components as NumPy arrays."""
    f_input = ROOT.TFile.Open(root_file, "READ")
    if not f_input or f_input.IsZombie():
        raise OSError(f"Non riesco ad aprire il file ROOT: {root_file}")

    try:
        nomi = {
            "bkg": f"h_{var_name}_bkg",
            "00": f"h_{var_name}_00",
            "tt": f"h_{var_name}_tt",
            "mixed": f"h_{var_name}_mixed",
        }
        hist = {}
        for chiave, nome in nomi.items():
            h = f_input.Get(nome)
            if not h:
                raise KeyError(f"Istogramma '{nome}' non trovato in {root_file}")
            hist[chiave] = h

        n_bins = hist["bkg"].GetNbinsX()
        for chiave, h in hist.items():
            if h.GetNbinsX() != n_bins:
                raise ValueError(
                    f"Numero di bin non coerente per '{nomi[chiave]}' "
                    f"({h.GetNbinsX()} invece di {n_bins})"
                )

        arrays = {
            chiave: np.array(
                [h.GetBinContent(i) for i in range(1, n_bins + 1)], dtype=float
            )
            for chiave, h in hist.items()
        }
    finally:
        f_input.Close()

    return arrays


def crea_nll(template):
    """Build the Poisson deviance (-2 Delta log L) for the supplied templates."""
    bkg = template["bkg"]
    sig00 = template["00"]
    sigtt = template["tt"]
    sigmix = template["mixed"]

    
    data = bkg + sig00 + sigtt + sigmix
    mask_data = data > 0

    def negative_log_likelihood(mu_00, mu_tt, mu_mix):
        nu = bkg + mu_00 * sig00 + mu_tt * sigtt + mu_mix * sigmix
        if np.any((nu <= 0) & mask_data):
            return 1e100

        term_data = np.zeros_like(data)
        term_data[mask_data] = data[mask_data] * np.log(data[mask_data] / nu[mask_data])
        return float(2.0 * np.sum(nu - data + term_data))

    return negative_log_likelihood


def esegui_fit_likelihood(var_name, parametri_liberi=PARAMS,
                          root_file=TEMPLATE_FILE, stampa=False):
    """Run the requested fit, fixing all parameters not selected as free to 1."""
    parametri_liberi = tuple(parametri_liberi)
    if not parametri_liberi:
        raise ValueError("Deve esserci almeno un parametro libero.")
    if any(p not in PARAMS for p in parametri_liberi):
        raise ValueError(f"Parametri non riconosciuti: {parametri_liberi}")

    nll = crea_nll(carica_template(var_name, root_file))
    m = Minuit(nll, mu_00=1.0, mu_tt=1.0, mu_mix=1.0)
    m.errordef = 1.0
    for p in PARAMS:
        #m.limits[p] = (0.0, 5.0)
        m.fixed[p] = p not in parametri_liberi
        if p not in parametri_liberi:
            m.values[p] = 1.0

    m.migrad()
    m.hesse()


    if stampa:
        print(f"Fit {len(parametri_liberi)}D — {var_name} — liberi: {', '.join(parametri_liberi)}")
        for p in PARAMS:
            if p in parametri_liberi:
                print(f"  {NOMI_PARAMETRI[p]} = {m.values[p]:.5f} ± {m.errors[p]:.5f}")
            else:
                print(f"  {NOMI_PARAMETRI[p]} = 1 (fixed)")
        print(f"  valido={m.valid}; covarianza accurata={getattr(m.fmin, 'has_accurate_covar', None)}; EDM={m.fmin.edm:.3g}")

    return m


def leggi_correlazione(fit, p1, p2, parametri_liberi):
    """Extract a parameter correlation across supported iminuit versions."""
    corr = fit.covariance.correlation()
    try:
        return float(corr[p1, p2])
    except Exception:
        arr = np.asarray(corr, dtype=float)
        ordine = list(PARAMS) if arr.shape[0] == len(PARAMS) else list(parametri_liberi)
        return float(arr[ordine.index(p1), ordine.index(p2)])


def stampa_tabella_testo(titolo, righe, colonne):
    """Print an aligned text table when the code is run outside Jupyter."""
    print(f"\n{titolo}")
    if not righe:
        print("(no results)")
        return
    larghezze = {
        col: max(len(str(col)), *(len(str(r.get(col, ""))) for r in righe))
        for col in colonne
    }
    intestazione = " | ".join(str(col).ljust(larghezze[col]) for col in colonne)
    separatore = "-+-".join("-" * larghezze[col] for col in colonne)
    print(intestazione)
    print(separatore)
    for riga in righe:
        print(" | ".join(str(riga.get(col, "")).ljust(larghezze[col]) for col in colonne))


# Run one 1D fit for each parameter, three 2D fits for the parameter pairs, and one 3D fit
# for each observable. Individual fits suppress repetitive detailed output.
fit_results = {}
righe_riepilogo = []
righe_correlazioni = []

for var_name in VARIABILI:
    combinazioni_fit = [
        combo
        for n_liberi in (1, 2, 3)
        for combo in combinations(PARAMS, n_liberi)
    ]

    for parametri_liberi in combinazioni_fit:
        fit = esegui_fit_likelihood(var_name, parametri_liberi, stampa=False)
        fit_results[(var_name, tuple(parametri_liberi))] = fit

        dimensione = len(parametri_liberi)
        riga = {
            "Fit": f"{dimensione}D",
            "Parametri liberi": ", ".join(NOMI_PARAMETRI[p] for p in parametri_liberi),
            "μ_LL": (f"{fit.values['mu_00']:.5f} ± {fit.errors['mu_00']:.5f}"
                     if "mu_00" in parametri_liberi else "1 (fisso)"),
            "μ_TT": (f"{fit.values['mu_tt']:.5f} ± {fit.errors['mu_tt']:.5f}"
                     if "mu_tt" in parametri_liberi else "1 (fisso)"),
            "μ_LT": (f"{fit.values['mu_mix']:.5f} ± {fit.errors['mu_mix']:.5f}"
                     if "mu_mix" in parametri_liberi else "1 (fisso)"),
            "Valido": "OK" if fit.valid else "NO",
            "Cov. accurata": "sì" if getattr(fit.fmin, "has_accurate_covar", False) else "no",
            "EDM": f"{fit.fmin.edm:.2g}",
            "_variabile": var_name,
        }
        righe_riepilogo.append(riga)

        if dimensione >= 2 and fit.covariance is not None:
            for p1, p2 in combinations(parametri_liberi, 2):
                rho = leggi_correlazione(fit, p1, p2, parametri_liberi)
                righe_correlazioni.append({
                    "Fit": f"{dimensione}D",
                    "Coppia": f"ρ({NOMI_PARAMETRI[p1]}, {NOMI_PARAMETRI[p2]})",
                    "ρ": f"{rho:+.4f}",
                    "_variabile": var_name,
                })


fit_cos = fit_results[("cos_theta_star", PARAMS)]
fit_pt  = fit_results[("lep_pt", PARAMS)]
fit_mll = fit_results[("mll", PARAMS)]

COLONNE_FIT = ["Fit", "Parametri liberi", "μ_LL", "μ_TT", "μ_LT", "Valido", "Cov. accurata", "EDM"]
COLONNE_CORR = ["Fit", "Coppia", "ρ"]


COLONNE_FIT_ORDINATE = [
    "Variabile", "Parametri liberi", "μ_LL", "μ_TT", "μ_LT",
    "Valido", "Cov. accurata", "EDM"
]
COLONNE_CORR_ORDINATE = ["Variabile", "Coppia", "ρ"]


def righe_per_display(righe, nome_colonna_variabile="Variabile"):
    """Prepare result rows and convert observable keys into readable labels."""
    output = []
    for r in righe:
        item = {k: v for k, v in r.items() if k != "_variabile" and k != "Fit"}
        item[nome_colonna_variabile] = NOMI_VARIABILI[r["_variabile"]]
        output.append(item)
    return output


if IN_NOTEBOOK and pd is not None:
    for dimensione in (1, 2, 3):
        titolo = f"Fit {dimensione}D"
        display(Markdown(f"## {titolo}"))
        subset = [r for r in righe_riepilogo if r["Fit"] == f"{dimensione}D"]
        rows_display = righe_per_display(subset)
        display(pd.DataFrame(rows_display, columns=COLONNE_FIT_ORDINATE))

    display(Markdown("## Correlazioni"))
    for dimensione in (2, 3):
        display(Markdown(f"### Correlazioni dei fit {dimensione}D"))
        subset = [r for r in righe_correlazioni if r["Fit"] == f"{dimensione}D"]
        rows_display = righe_per_display(subset)
        display(pd.DataFrame(rows_display, columns=COLONNE_CORR_ORDINATE))
else:
    for dimensione in (1, 2, 3):
        subset = [r for r in righe_riepilogo if r["Fit"] == f"{dimensione}D"]
        stampa_tabella_testo(
            f"Fit {dimensione}D",
            righe_per_display(subset),
            COLONNE_FIT_ORDINATE,
        )

    print("\nCORRELAZIONI")
    for dimensione in (2, 3):
        subset = [r for r in righe_correlazioni if r["Fit"] == f"{dimensione}D"]
        stampa_tabella_testo(
            f"Correlazioni dei fit {dimensione}D",
            righe_per_display(subset),
            COLONNE_CORR_ORDINATE,
        )


riepilogo_fit = pd.DataFrame(righe_riepilogo) if pd is not None else righe_riepilogo
riepilogo_correlazioni = pd.DataFrame(righe_correlazioni) if pd is not None else righe_correlazioni


In [ ]:
# One- and two-dimensional likelihood scans

from itertools import combinations
import os
from IPython.display import display, Image

# Detect whether the code is running inside Jupyter.
try:
    from IPython import get_ipython
    IN_NOTEBOOK = get_ipython() is not None
except Exception:
    IN_NOTEBOOK = False

# PNG files are saved to disk as well as displayed in the notebook.
CARTELLA_GRAFICI = output_dir("likelihood_plots")

def mostra_canvas(canvas, nome_file):
    """Update and save a ROOT canvas; display the PNG inline in Jupyter."""
    percorso = os.path.join(CARTELLA_GRAFICI, nome_file)
    canvas.Modified()
    canvas.Update()
    canvas.SaveAs(percorso)

    if IN_NOTEBOOK:
        display(Image(filename=percorso))
    else:
        canvas.Draw()

VARIABILI_SCAN = VARIABILI          # per farne solo alcune: ("cos_theta_star",)

# Parameter names in ROOT format (for axes and legends).
LATEX_PARAMETRI = {"mu_00": "#mu_{LL}", "mu_tt": "#mu_{TT}", "mu_mix": "#mu_{LT}"}

tieni_vivi = []     # canvas e grafici vanno tenuti in una lista, altrimenti Python li cancella


def punti_buoni(x, y, ok=None):
    """Remove scan points where the fit failed (large objective value or ok = False)."""
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    buoni = y < 100
    if ok is not None:
        buoni = buoni & np.array(ok, dtype=bool)
    return x[buoni], y[buoni]


# ============================================================
# One-dimensional scan for a single parameter.
# ============================================================
def scan_1d(var_name, param):
    fit_fisso = fit_results[(var_name, (param,))]     # altri parametri fissi a 1
    fit_prof = fit_results[(var_name, PARAMS)]        # altri parametri profilati

    # Use the same range for both curves: plus/minus three standard deviations from the 3D fit.
    val, err = fit_prof.values[param], fit_prof.errors[param]
    intervallo = (val - 3 * err, val + 3 * err)

    # profile: evaluate the function with the other parameters fixed to their nominal value (= 1).
    r = fit_fisso.profile(param, size=40, bound=intervallo, subtract_min=True)
    x_f, y_f = punti_buoni(r[0], r[1])

    # mnprofile: refit at each point while profiling the other two parameters.
    x_p, y_p, ok_p = fit_prof.mnprofile(param, size=40, bound=intervallo, subtract_min=True)
    x_p, y_p = punti_buoni(x_p, y_p, ok_p)

    c = make_canvas(f"c_scan1d_{var_name}_{param}", "Likelihood Scan 1D", 900, 650)
    c.SetTopMargin(0.20)            # spazio sopra il grafico per la legenda
    c.cd()

    g_p = ROOT.TGraph(len(x_p), x_p, y_p)
    g_p.SetTitle(f";{LATEX_PARAMETRI[param]};-2 #Delta ln L")
    g_p.SetLineColor(COLORS_FIT["profilato"])
    g_p.SetLineWidth(LINE_WIDTH)
    g_p.SetMinimum(0)               # asse y da 0 a 9: si vedono le soglie di 1 e 2 sigma
    g_p.SetMaximum(9)
    g_p.Draw("AL")
    g_p.GetXaxis().SetLimits(intervallo[0], intervallo[1])
    g_p.GetXaxis().SetTitleOffset(1.4)
    g_p.GetYaxis().SetTitleOffset(1.4)

    g_f = ROOT.TGraph(len(x_f), x_f, y_f)
    g_f.SetLineColor(COLORS_FIT["fisso"])
    g_f.SetLineWidth(LINE_WIDTH)
    g_f.SetLineStyle(2)
    g_f.Draw("L")

    # One- and two-sigma thresholds (Delta = 1 and Delta = 4).
    linea_1s = ROOT.TLine(intervallo[0], 1.0, intervallo[1], 1.0)
    linea_1s.SetLineColor(COLORS_FIT["soglie"])
    linea_1s.SetLineStyle(2)
    linea_1s.Draw()
    linea_2s = ROOT.TLine(intervallo[0], 4.0, intervallo[1], 4.0)
    linea_2s.SetLineColor(COLORS_FIT["soglie"])
    linea_2s.SetLineStyle(3)
    linea_2s.Draw()

    leg = make_legend(0.20, 0.81, 0.95, 0.91)
    leg.SetNColumns(2)
    leg.AddEntry(g_p, "Others profiled (3D fit)", "l")
    leg.AddEntry(g_f, "Others fixed to 1", "l")
    leg.AddEntry(linea_1s, "1#sigma (#Delta = 1)", "l")
    leg.AddEntry(linea_2s, "2#sigma (#Delta = 4)", "l")
    leg.Draw()

    draw_fcc_label()
    mostra_canvas(c, f"scan1d_{var_name}_{param}.png")
    tieni_vivi.extend([c, g_p, g_f, linea_1s, linea_2s, leg])


# ============================================================
# Two-dimensional confidence-contour scan for a pair of parameters.
# ============================================================
def scan_2d(var_name, px, py):
    terzo = [p for p in PARAMS if p not in (px, py)][0]
    fit_fisso = fit_results[(var_name, (px, py))]     
    fit_prof = fit_results[(var_name, PARAMS)]        
    nome_terzo = LATEX_PARAMETRI[terzo]

    contorni = [
        # Profiled contours: blue, with a light fill for the 95% confidence level.
        (fit_prof, 0.95, COLORS_FIT["profilato"], 1, True,
         f"95% CL, {nome_terzo} profiled", COLORS_FIT["profilato_chiaro"]),
        (fit_prof, 0.68, COLORS_FIT["profilato"], 1, True,
         f"68% CL, {nome_terzo} profiled", COLORS_FIT["profilato"]),
        # Contours with the third parameter fixed: grayscale.
        (fit_fisso, 0.95, COLORS_FIT["fisso_chiaro"], 2, False,
         f"95% CL, {nome_terzo} = 1", None),
        (fit_fisso, 0.68, COLORS_FIT["fisso"], 2, False,
         f"68% CL, {nome_terzo} = 1", None),
    ]

    c = make_canvas(f"c_scan2d_{var_name}_{px}_{py}", "Likelihood Scan 2D", 900, 650)
    c.SetTopMargin(0.20)            # spazio sopra il grafico per la legenda
    c.cd()

    leg = make_legend(0.20, 0.81, 0.95, 0.91)
    leg.SetNColumns(2)
    grafici = []

    for i, (fit, cl, colore, stile, riempi, etichetta, colore_riempimento) in enumerate(contorni):
        # mncontour returns the (x, y) points of the contour.
        punti = np.array(fit.mncontour(px, py, cl=cl, size=60), dtype=float)
        # mncontour does not close the curve; append the first point to close it.
        x_c = np.append(punti[:, 0], punti[0, 0])
        y_c = np.append(punti[:, 1], punti[0, 1])

        g = ROOT.TGraph(len(x_c), x_c, y_c)
        g.SetLineColor(colore)
        g.SetLineWidth(LINE_WIDTH)
        g.SetLineStyle(stile)
        if riempi:
            g.SetFillColorAlpha(colore_riempimento, 0.25)
        grafici.append(g)

        opzione = "LF" if riempi else "L"
        if i == 0:
            g.SetTitle(f";{LATEX_PARAMETRI[px]};{LATEX_PARAMETRI[py]}")
            g.Draw("A" + opzione)
            g.GetXaxis().SetTitleOffset(1.4)
            g.GetYaxis().SetTitleOffset(1.4)
        else:
            g.Draw(opzione)
        leg.AddEntry(g, etichetta, "lf" if riempi else "l")

    # Best-fit point.
    g_best = ROOT.TGraph(1, np.array([fit_prof.values[px]], dtype=float),
                            np.array([fit_prof.values[py]], dtype=float))
    g_best.SetMarkerStyle(20)
    g_best.SetMarkerSize(1.2)
    g_best.SetMarkerColor(COLORS_FIT["fisso"])
    g_best.Draw("P")
    leg.AddEntry(g_best, "Best fit", "p")

    leg.Draw()
    draw_fcc_label()
    mostra_canvas(c, f"scan2d_{var_name}_{px}_{py}.png")
    tieni_vivi.extend([c, leg, g_best] + grafici)


# ============================================================
# Loop over observables, parameters, and parameter pairs.
# ============================================================
for var_name in VARIABILI_SCAN:
    print(f"\n===== {NOMI_VARIABILI[var_name]} =====")

    for param in PARAMS:                            # 3 scan 1D
        try:
            scan_1d(var_name, param)
        except Exception as errore:
            print(f"[!] scan 1D saltato ({var_name}, {param}): {errore}")

    for px, py in combinations(PARAMS, 2):          # 3 scan 2D
        try:
            scan_2d(var_name, px, py)
        except Exception as errore:
            print(f"[!] scan 2D saltato ({var_name}, {px}, {py}): {errore}")


In [ ]:
# Save all canvases as vector PDF and PNG files
# PDF preserves vector quality when figures are resized.
# Run this cell after creating the canvases to export all figures.
import os
for canvas in ROOT.gROOT.GetListOfCanvases():
    canvas.SaveAs(output_path(f"plots/{canvas.GetName()}.pdf"))
    canvas.SaveAs(output_path(f"plots/{canvas.GetName()}.png"))
